# Object Detection
This notebook handles live sensor streaming from an Intel RealSense depth camera, object detection training, and 3D spatial coordinate projection.

**Goal: Detect pallets in an industrial environment and estimate their 3D position.**

**Steps:**
1. Record **xxx** images containing pallets in an industrial environment.
2. Label images. Draw bounding boxes around pallets. Export labels in Darknet (YOLO) format.
3. Train YOLO model.
4. Analyze performances.
5. Run live inference

## 1. Record images using Realsense camera.
The frame rate is set to 5 frames per second to increase the variety of scenarios. For this reason, the display may not appear smooth.

In [ ]:
import cv2
import numpy as np
import pyrealsense2 as rs

# Initialize the pipeline configuration
pipeline = rs.pipeline()
config = rs.config()
framerate = 5 # FPS
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, framerate)
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, framerate)

pipeline.start(config)

try:
    while True:
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()

        if not color_frame or not depth_frame:
            continue

        # Convert frame data to numpy arrays
        color_image = np.asanyarray(color_frame.get_data())

        cv2.imshow("Live RealSense Color Feed Stream", color_image)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
finally:
    pipeline.stop()
    cv2.destroyAllWindows()

## 2. Label pallets using [CVAT](https://www.cvat.ai/).
- Create bounding boxes around pallets.
- Export labels to Darknet / YOLO format.
- Split data into train / validation sets.
- Move this dataset into `datasets/pallet_detection` directory of this project.
- Create `datasets/pallet_detection/dataset.yaml` file:
    ```yaml
    path: ./datasets/pallet_detection/
    train: images/train
    val: images/val
    names:
      0: pallet
    ```

## 3. Train custom YOLO model

In [ ]:
from ultralytics import YOLO

# Load a lightweight pre-trained YOLO model
model = YOLO("yolov8s.pt")

# Run single-class fine-tuning optimization
model.train(data="./datasets/pallet_detection/dataset.yaml", epochs=15, imgsz=640, classes=[0])

## 4. Realtime inference visualization

In [ ]:
from ultralytics import YOLO
import pyrealsense2 as rs
import numpy as np
import cv2

model = YOLO("runs/detect/train/weights/best.pt")
pipeline = rs.pipeline()
config = rs.config()
framerate = 10 # FPS
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, framerate)
config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, framerate)

profile = pipeline.start(config)
# Retrieve camera intrinsic parameters needed for 3D projection mapping
intrinsics = profile.get_stream(rs.stream.color).as_video_stream_profile().get_intrinsics()

try:
    while True:
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        depth_frame = frames.get_depth_frame()
        if not color_frame or not depth_frame: continue

        color_img = np.asanyarray(color_frame.get_data())

        # Run inference loop step
        results = model(color_img, verbose=False)[0]

        for box in results.boxes:
            # Extract bounding box edges
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # TODO: Compute pixel coordinate centers of the bounding boxes
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            # Fetch measured raw depth distance at the center pixel location
            depth_distance = depth_frame.get_distance(cx, cy)

            if depth_distance > 0:
                # Transform 2D pixel coordinates + depth into 3D spatial points (X, Y, Z)
                point_3d = rs.rs2_deproject_pixel_to_point(intrinsics, [cx, cy], depth_distance)
                X, Y, Z = point_3d

                # Draw visual elements on the image frame
                cv2.rectangle(color_img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.circle(color_img, (cx, cy), 5, (0, 0, 255), -1)
                cv2.putText(color_img, f"XYZ: {X:.2f}m, {Y:.2f}m, {Z:.2f}m",
                            (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

        cv2.imshow("Spatial 3D Tracking Detection Loop Feed", color_img)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
finally:
    pipeline.stop()
    cv2.destroyAllWindows()